# Algorithmic Fairness, Session 2: Mitigation and Trade-offs
**HON-2501-3 · Responsible AI** · class notebook

<a href="https://colab.research.google.com/github/acintron-cc/hon-2501-03/blob/main/notebooks/fairness_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Short link: tinyurl.com/ya84k9yy. You don't write any code: you choose values in form fields and press ▶.

**How to use this notebook**
1. **File → Save a copy in Drive**, so your work is kept.
2. In **Section 2**, choose your dataset from the dropdown.
3. Run the cells one at a time with ▶, from the top.
4. When a cell has a form field (a dropdown, a slider or a text box), change the value there and press ▶ on that cell.

> **Gemini rule:** Ask Gemini to explain a cell or a number. Never ask it to rewrite or fix a cell.

| Class | Sections |
|---|---|
| Class 1 | 1–4: setup, dataset, metrics by group, reweighing |
| Class 2 | 5–6: thresholds, compare three models |

## Section 1 · Setup
This cell installs Fairlearn, the fairness library we use, and loads the other tools. It prints Ready. when it is done (1 to 2 minutes the first time). If Colab warns "This notebook was not authored by Google", choose Run anyway.

In [ ]:
# @title Install and load the tools
!pip install -q fairlearn==0.13.0

import ast, math, os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate, false_positive_rate, count
from fairlearn.postprocessing import ThresholdOptimizer
warnings.filterwarnings("ignore")

DATA_URL = "https://raw.githubusercontent.com/acintron-cc/hon-2501-03/main/datasets/"
TEAL, ORANGE = "#0B6E84", "#A8401A"
RATES = ["selection rate", "TPR", "FPR", "accuracy"]
METRICS = {
    "n": count,
    "selection rate": selection_rate,
    "TPR": true_positive_rate,
    "FPR": false_positive_rate,
    "accuracy": accuracy_score,
}
GAP_ROW = "gap (larger − smaller)"
GAPS = {"selection rate": "selection-rate gap", "TPR": "TPR gap", "FPR": "FPR gap"}

def rnd(x, d=3):
    """Round half up, the way Excel does."""
    return math.floor(x * 10**d + 0.5 + 1e-9) / 10**d

def fmt(x, d=3):
    return f"{rnd(x, d):.{d}f}"

def group_line():
    print(f"Group A = {GROUPS[0]}, Group B = {GROUPS[1]}")

def two_way(rows):
    """Group x label counts with totals."""
    t = pd.crosstab(rows[GROUP], rows["label"]).reindex(index=GROUPS, columns=[1, 0], fill_value=0)
    t.columns = ["label = 1", "label = 0"]
    t["Total"] = t.sum(axis=1)
    t.loc["Total"] = t.sum()
    t.index.name = GROUP
    return t

def by_group(y_true, y_pred, sensitive):
    """Fairlearn's MetricFrame: each metric computed separately for each group."""
    mf = MetricFrame(metrics=METRICS, y_true=y_true, y_pred=y_pred, sensitive_features=sensitive)
    return mf.by_group.reindex(GROUPS)

def gap(col):
    """Gap = larger group value - smaller group value, naming the larger group."""
    a, b = col[GROUPS[0]], col[GROUPS[1]]
    g = rnd(abs(a - b))
    if g == 0:
        return "0.000 (equal)"
    return f"{g:.3f} ({GROUPS[0] if a > b else GROUPS[1]} higher)"

def gap_size(bg, c):
    return rnd(abs(bg.loc[GROUPS[0], c] - bg.loc[GROUPS[1], c]))

def rate_table(bg, cols=RATES):
    out = bg[cols].map(fmt)
    out.loc[GAP_ROW] = [gap(bg[c]) for c in cols]
    return out

def counts(rows, pred):
    y, p = rows["label"].to_numpy(), np.asarray(pred)
    return dict(TP=int(((y == 1) & (p == 1)).sum()), FN=int(((y == 1) & (p == 0)).sum()),
                FP=int(((y == 0) & (p == 1)).sum()), TN=int(((y == 0) & (p == 0)).sum()))

CALC_OPS = {ast.Add: lambda a, b: a + b, ast.Sub: lambda a, b: a - b,
            ast.Mult: lambda a, b: a * b, ast.Div: lambda a, b: a / b,
            ast.UAdd: lambda a: a, ast.USub: lambda a: -a}

def safe_calc(text):
    """Evaluate numbers with + - * / and parentheses only (no eval). Returns a number or a message."""
    friendly = ("The calculator only understands numbers, + - * / and parentheses, "
                "for example (343/1400) / (55/466). Please check what you typed and press ▶ again.")
    if not re.fullmatch(r"[0-9\s.+\-*/()]+", text or ""):
        return friendly
    def ev(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in CALC_OPS:
            return CALC_OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in CALC_OPS:
            return CALC_OPS[type(node.op)](ev(node.operand))
        raise ValueError
    try:
        return ev(ast.parse(text, mode="eval").body)
    except ZeroDivisionError:
        return "That divides by zero. Please check your fractions and press ▶ again."
    except (SyntaxError, ValueError):
        return friendly

def exact_weights():
    """Reference weights, P(label) / P(label | group), computed from the train-row fractions."""
    p1 = train["label"].mean()
    w = {}
    for g in GROUPS:
        pg = train.loc[train[GROUP] == g, "label"].mean()
        w[(g, 1)], w[(g, 0)] = p1 / pg, (1 - p1) / (1 - pg)
    return w

def your_weights():
    """The class notebook's four weight fields (Section 4.2)."""
    A, B = GROUPS
    v = globals()
    return {(A, 1): v.get("w_label1_groupA", 0.0), (A, 0): v.get("w_label0_groupA", 0.0),
            (B, 1): v.get("w_label1_groupB", 0.0), (B, 0): v.get("w_label0_groupB", 0.0)}

def check_weights(w):
    """For each group: (group, arithmetic, weighted rate of label = 1, passes?)."""
    p1 = train["label"].mean()
    out = []
    for g in GROUPS:
        t = train[train[GROUP] == g]
        n1 = int((t["label"] == 1).sum())
        n0 = len(t) - n1
        top = n1 * w[(g, 1)]
        bottom = top + n0 * w[(g, 0)]
        rate = top / bottom
        text = (f"({n1} × {w[(g, 1)]:.4f}) ÷ ({n1} × {w[(g, 1)]:.4f} + {n0} × {w[(g, 0)]:.4f})"
                f" = {fmt(top, 2)} ÷ {fmt(bottom, 2)} = {fmt(rate, 4)}")
        out.append((g, text, rate, abs(rate - p1) <= 0.005))
    return out

def print_check(w):
    n1, N = int((train["label"] == 1).sum()), len(train)
    print(f"P(label = 1) = {n1}/{N} = {fmt(n1 / N, 4)}\n")
    for g, text, rate, ok in check_weights(w):
        print(f"Weighted rate of label = 1, {g}:")
        print(f"  {text}")
        print(f"  Same as P(label = 1)? {'yes' if ok else 'recheck'}\n")

def fit_reweighed(w):
    """Retrain the original model type with each train row counting by its weight; predict the test rows."""
    rw = np.array([w[(g, y)] for g, y in zip(train[GROUP], train["label"])])
    m = new_model()
    m.fit(X_train, y_train, logisticregression__sample_weight=rw)
    return m.predict(X_test)

def fit_threshold(constraint):
    """Fairlearn's ThresholdOptimizer on the train rows; predictions for the train and test rows."""
    to = ThresholdOptimizer(estimator=new_model(), constraints=constraint, objective="accuracy_score",
                            prefit=False, predict_method="predict_proba")
    to.fit(X_train, y_train, sensitive_features=train[GROUP])
    return (np.asarray(to.predict(X_train, sensitive_features=train[GROUP], random_state=42)),
            np.asarray(to.predict(X_test, sensitive_features=test[GROUP], random_state=42)))

def before_after(before, after):
    """Rates and gaps before and after, side by side."""
    cols = {}
    for g in GROUPS:
        cols[f"{g}, before"] = before.loc[g, RATES].map(fmt)
        cols[f"{g}, after"] = after.loc[g, RATES].map(fmt)
    cols["gap, before"] = [gap(before[c]) for c in RATES]
    cols["gap, after"] = [gap(after[c]) for c in RATES]
    return pd.DataFrame(cols, index=RATES)

def threshold_game(cuts):
    """One cutoff per group on the model's score (test rows): predictions, rates and the results table."""
    pred = (test["score"] >= test[GROUP].map(cuts)).astype(int)
    bg = by_group(y_test, pred, test[GROUP])
    out = rate_table(bg)
    out.insert(0, "cutoff", [f"{cuts[g]:.2f}" for g in GROUPS] + [""])
    return pred, bg, out

def compare(models):
    """Rates and gaps of several models (test rows); gap cells colored against the first model."""
    first = list(models)[0]
    table, gaps = {}, {}
    for name, p in models.items():
        bg_m = by_group(y_test, p, test[GROUP])
        row = {f"{c}, {g}": fmt(bg_m.loc[g, c]) for c in GAPS for g in GROUPS}
        gaps[name] = {c: gap_size(bg_m, c) for c in GAPS}
        for c, label in GAPS.items():
            row[label] = gap(bg_m[c])
            if name != first:
                d = gaps[name][c] - gaps[first][c]
                row[label] += " ↓" if d < 0 else " ↑" if d > 0 else ""
        row["accuracy, all"] = fmt(accuracy_score(y_test, p))
        table[name] = row
    table = pd.DataFrame(table).T

    def color_gaps(t):
        styles = pd.DataFrame("", index=t.index, columns=t.columns)
        for name in list(models)[1:]:
            for c, label in GAPS.items():
                d = gaps[name][c] - gaps[first][c]
                if d != 0:
                    styles.loc[name, label] = f"background-color: {TEAL if d < 0 else ORANGE}; color: white; font-weight: bold"
        return styles

    return table, gaps, table.style.apply(color_gaps, axis=None)

def plot_gaps(gaps):
    """One panel per gap; the first model is the outlined reference bar."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True)
    names = list(gaps)
    top = max(max(gaps[m].values()) for m in names)
    for ax, (c, label) in zip(axes, GAPS.items()):
        for i, m in enumerate(names):
            v, d = gaps[m][c], gaps[m][c] - gaps[names[0]][c]
            if i == 0:
                ax.bar(i, v, 0.6, color="white", edgecolor=TEAL, linewidth=2)
                note = names[0]
            else:
                ax.bar(i, v, 0.6, color=ORANGE if d > 0 else TEAL)
                note = "larger" if d > 0 else "smaller" if d < 0 else "same"
            ax.text(i, v + top * 0.03, f"{v:.3f}\n{note}", ha="center", va="bottom", fontsize=9)
        ax.set_xticks(range(len(names)), [m.split(" (")[0] for m in names])
        ax.set_title(label)
        ax.set_xlabel("model")
        ax.grid(axis="y", alpha=0.3)
    axes[0].set_ylabel("gap (larger − smaller)")
    axes[0].set_ylim(0, top * 1.35 + 0.01)
    fig.suptitle(f"{DATASET}: gaps between {GROUPS[0]} and {GROUPS[1]}, test rows (smaller = more equal)")
    fig.tight_layout()
    return fig

GAP_LEGEND = "Teal with ↓: the gap is smaller than the original model's.  Orange with ↑: the gap is larger.  No color: the same gap."

print("Ready.")

## Section 2 · Choose your dataset
Pick your dataset from the dropdown, then press ▶. Each dataset compares two groups, called Group A and Group B below.

In [ ]:
# @title Choose your dataset
DATASET = "adult"  # @param ["adult", "compas", "german_credit", "bank"]

CONFIG = {
    "adult": dict(target="class", group="sex", groups=["Female", "Male"],
                  drop=["fnlwgt", "education", "sex", "race"],
                  meaning="income over $50K", favorable=True),
    "compas": dict(target="two_year_recid", group="race_group", groups=["African-American", "Caucasian"],
                   drop=["race", "sex", "c_charge_desc"],
                   meaning="rearrested within 2 years", favorable=False),
    "german_credit": dict(target="credit-risk", group="sex", groups=["female", "male"],
                          drop=["sex", "marital_status", "age"],
                          meaning="good credit risk", favorable=True),
    "bank": dict(target="deposit", group="age_group", groups=["under 25 or 60+", "25-59"],
                 drop=["age"], meaning="subscribed to a deposit", favorable=True),
}
cfg = CONFIG[DATASET]
GROUP, GROUPS = cfg["group"], cfg["groups"]

local = os.environ.get("FAIRNESS_LOCAL_DATA")   # instructor's offline tests only
df = pd.read_csv(os.path.join(local, f"{DATASET}_classroom.csv") if local else DATA_URL + f"{DATASET}_classroom.csv")
df = df[df[GROUP].isin(GROUPS)].reset_index(drop=True)
train = df[df["split"] == "train"]
test = df[df["split"] == "test"]

# The model: a logistic regression that never sees the group column.
never = ["label", "split", "score", "predicted", "predicted_reweighed", "predicted_threshold",
         cfg["target"], GROUP] + cfg["drop"]
FEATURES = [c for c in df.columns if c not in never]
CATEGORICAL = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(df[c])]
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]

def new_model():
    return make_pipeline(
        make_column_transformer((OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
                                (StandardScaler(), NUMERIC)),
        LogisticRegression(max_iter=1000))

X_train, y_train = train[FEATURES], train["label"]
X_test, y_test = test[FEATURES], test["label"]
pred_ref_rw = pred_ref_to = pred_ref_to_train = None   # reference models, built in Sections 4 and 5

print(f"Dataset: {DATASET}")
print(f"label = 1 means: {cfg['meaning']}. This is a {'favorable' if cfg['favorable'] else 'HARMFUL'} outcome for the person.")
print(f"Group column: {GROUP}")
group_line()
if DATASET == "compas":
    print("Rows with race_group = Other are left out.")
print(f"Rows: {len(train):,} train, {len(test):,} test")

## Section 3 · Metrics by group (Exercise 3; Class 1 starts here)
### 3.1 · Base rates, all rows
The base rate is P(label = 1 | group): how often each group has label = 1, before any model is involved. The bar | means "among".

In [ ]:
# @title Base rates, all rows
t = two_way(df)
display(t)
for g in GROUPS:
    a, b = t.loc[g, "label = 1"], t.loc[g, "Total"]
    print(f"P(label = 1 | {g}) = {a}/{b} = {fmt(a / b)}")

### 3.2 · Confusion matrix per group, test rows
A confusion matrix counts the test rows by what really happened (actual) and what the model said (predicted). TP = true positive, FN = false negative, FP = false positive, TN = true negative.

In [ ]:
# @title Confusion matrix per group, test rows
for g in GROUPS:
    t = test[test[GROUP] == g]
    c = counts(t, t["predicted"])
    cm = pd.DataFrame([[f"TP = {c['TP']}", f"FN = {c['FN']}"], [f"FP = {c['FP']}", f"TN = {c['TN']}"]],
                      index=["actual = 1", "actual = 0"], columns=["predicted = 1", "predicted = 0"])
    print(f"\n{g} ({len(t)} test rows)")
    display(cm)

### 3.3 · Rates and gaps, test rows
Each rate is a fraction built from the counts in 3.2. The gap is the larger group's value minus the smaller group's value. `MetricFrame` is the Fairlearn tool that computes each rate for each group.

In [ ]:
# @title Rates and gaps, test rows
bg = by_group(y_test, test["predicted"], test[GROUP])
rates = bg[RATES].map(fmt)
rates.insert(0, "n", bg["n"].astype(int))
print("Rates by group (MetricFrame):")
display(rates)

fractions, selected = {}, {}
for g in GROUPS:
    t = test[test[GROUP] == g]
    c, n = counts(t, t["predicted"]), len(t)
    parts = {"selection rate": (c["TP"] + c["FP"], n), "TPR": (c["TP"], c["TP"] + c["FN"]),
             "FPR": (c["FP"], c["FP"] + c["TN"]), "accuracy": (c["TP"] + c["TN"], n)}
    fractions[g] = {k: f"{k} = {a}/{b} = {fmt(a / b)}" for k, (a, b) in parts.items()}
    selected[g] = parts["selection rate"]
print("\nThe same rates as fractions of the counts above:")
display(pd.DataFrame(fractions).T)

print("\nGaps:")
display(pd.DataFrame([[gap(bg[c]) for c in RATES]], index=[GAP_ROW], columns=RATES))

small, large = sorted((a / b, f"{a}/{b}") for a, b in selected.values())
ratio = small[0] / large[0]
print(f"\n80% rule: smaller selection rate ÷ larger = ({small[1]}) ÷ ({large[1]}) = {fmt(ratio)}")
print("Below 0.80: this flags a possible problem." if ratio < 0.8 else "0.80 or above: no flag.")
if not cfg["favorable"]:
    print("Remember: in COMPAS, being selected (predicted = 1) means being flagged as likely to be rearrested.")

In [ ]:
# @title Chart: rates by group, test rows
fig, ax = plt.subplots(figsize=(10.5, 4.5))
x = np.arange(len(RATES))
for i, (g, color) in enumerate(zip(GROUPS, [TEAL, ORANGE])):
    vals = bg.loc[g, RATES].to_numpy(dtype=float)
    bars = ax.bar(x + (i - 0.5) * 0.38, vals, 0.36, color=color)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{g}\n{fmt(v)}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x, RATES)
ax.set_ylim(0, 1.25)
ax.set_xlabel("rate (test rows)")
ax.set_ylabel("value (0 to 1)")
ax.set_title(f"{DATASET}: original model, by group")
ax.grid(axis="y", alpha=0.3)
plt.show()

**Exercise 3 questions** (copied from your handout). Double-click the empty text cell under each question to type your answer.

3a. For each group, write the TPR as a sentence: “Among \_\_\_ who really \_\_\_, the model said yes to \_\_\_%.”

3b. Which definition does your model meet (gap of 0.05 or less): demographic parity (equal selection rates), equal opportunity (equal TPR), equalized odds (equal TPR and FPR), or none?

3c. Which group has the higher accuracy? Does that make the model fairer to that group? Use your base rates from Exercise 1.

## Section 4 · Reweighing (Exercise 4; Class 1)
Reweighing gives each train row a weight, so that rare combinations of outcome and group count more: weight = P(label) ÷ P(label | group). You compute the four weights yourself.

### 4.1 · The train rows' two-way table
These counts use the train rows only, because the model learns from them. The second table shows the two fractions for each combination, and the field in 4.2 where its weight goes.

In [ ]:
# @title Train rows: two-way table and fractions
group_line()
t = two_way(train)
display(t)
N = t.loc["Total", "Total"]
rows = []
for g, letter in zip(GROUPS, "AB"):
    for y in (1, 0):
        rows.append({"combination": f"label = {y}, {g}",
                     "P(label | group)": f"{t.loc[g, f'label = {y}']}/{t.loc[g, 'Total']}",
                     "P(label)": f"{t.loc['Total', f'label = {y}']}/{N}",
                     "field in 4.2": f"w_label{y}_group{letter}"})
print("\nweight = P(label) ÷ P(label | group)")
display(pd.DataFrame(rows).set_index("combination"))

### 4.2 · Calculator, your weights, check
Type P(label) ÷ P(label | group) as fractions, for example (343/1400) / (55/466), and press ▶. Repeat for each of the four combinations in 4.1.

In [ ]:
# @title Calculator
EXPRESSION = "(343/1400) / (55/466)"  # @param {type:"string"}
result = safe_calc(EXPRESSION)
print(result if isinstance(result, str) else f"{EXPRESSION} = {result:.4f}")

Type your four weights (4 decimals) into the fields and press ▶. The output shows which combination each field stands for.

In [ ]:
# @title Your weights
w_label1_groupA = 0.0  # @param {type:"number"}
w_label0_groupA = 0.0  # @param {type:"number"}
w_label1_groupB = 0.0  # @param {type:"number"}
w_label0_groupB = 0.0  # @param {type:"number"}

group_line()
w = your_weights()
for (g, y), letter in zip(w, ["A", "A", "B", "B"]):
    print(f"label = {y}, {g}:  w_label{y}_group{letter} = {w[(g, y)]:.4f}")

Press ▶ to check your weights. After weighting, each group's rate of label = 1 should equal P(label = 1) on the train rows.

In [ ]:
# @title Check your weights
group_line()
w = your_weights()
if any(v == 0 for v in w.values()):
    print("Type your four weights in 4.2 first.")
else:
    print_check(w)

### 4.3 · Retrain with your weights
This trains the same model again on the train rows, with each row counting by its weight. The model never sees the group column. Press ▶ and compare the gaps with Section 3.3.

In [ ]:
# @title Retrain with your weights, test rows
pred_ref_rw = fit_reweighed(exact_weights())   # reference reweighed model, used in Section 6
w = your_weights()
if any(v == 0 for v in w.values()):
    print("Type your four weights in 4.2 first.")
elif not all(ok for *_, ok in check_weights(w)):
    print("Your weights don't pass the check in 4.2 yet. Fix the group marked recheck, then press ▶ here again.")
else:
    pred_yours = fit_reweighed(w)
    before = by_group(y_test, test["predicted"], test[GROUP])
    after = by_group(y_test, pred_yours, test[GROUP])
    print("Before = original model (Section 3.3). After = your reweighed model. Test rows.")
    display(before_after(before, after))
    agree = np.mean(pred_yours == pred_ref_rw)
    if agree < 1:
        print(f"Your model agrees with the reference model on {agree:.1%} of test rows.")

**Class 1 exit question.** In your dataset, which gaps did reweighing change, and in which direction? One sentence, with two numbers from Section 4.3.

**Next class:** reopen your saved copy and run Sections 1, 2 and 4 again (your weights are kept).

## Section 5 · Thresholds (Exercise 5; Class 2)
The model's scores stay the same; only the cutoff changes. A test row gets predicted = 1 when its score is at or above its group's cutoff.

### 5.1 · The threshold game, test rows
Round 1: move one group's cutoff until the selection-rate gap is 0.03 or less. Round 2: start again from 0.50 and make the TPR gap 0.03 or less. In the table (double-click it to type), record each round's cutoffs and rates, and which rate broke when you fixed the other, or, if none did, who paid for it (more people flagged, or more people missed).

| Round | Cutoff A | Cutoff B | Selection rate A | Selection rate B | TPR A | TPR B | FPR A | FPR B | What broke, or who paid |
|---|---|---|---|---|---|---|---|---|---|
| 1 | | | | | | | | | |
| 2 | | | | | | | | | |

In [ ]:
# @title Threshold game
cutoff_groupA = 0.50  # @param {type:"slider", min:0.01, max:0.99, step:0.01}
cutoff_groupB = 0.50  # @param {type:"slider", min:0.01, max:0.99, step:0.01}

group_line()
cuts = {GROUPS[0]: cutoff_groupA, GROUPS[1]: cutoff_groupB}
pred_game, bg_game, out = threshold_game(cuts)
display(out)

for name, c, label in [("Round 1", "selection rate", "selection-rate gap"), ("Round 2", "TPR", "TPR gap")]:
    g = gap_size(bg_game, c)
    print(f"{name} ({label} ≤ 0.03?): {'yes' if g <= 0.03 else 'not yet'}, gap = {g:.3f}")

### 5.2 · Fairlearn's `ThresholdOptimizer` (optional)
It tries cutoffs for each group and keeps the most accurate pair that meets a fairness constraint, a rule such as equal TPR. It is fitted on the train rows, so compare its train and test results.

In [ ]:
# @title ThresholdOptimizer
CONSTRAINT = "true_positive_rate_parity"  # @param ["true_positive_rate_parity", "demographic_parity", "false_positive_rate_parity", "equalized_odds"]

p_train, p_test = fit_threshold(CONSTRAINT)
print(f"Constraint: {CONSTRAINT}")
print("\nTrain rows (where the constraint is met):")
display(rate_table(by_group(y_train, p_train, train[GROUP])))
print("\nTest rows (new data):")
display(rate_table(by_group(y_test, p_test, test[GROUP])))

# Reference threshold model for Section 6: always equal TPR.
if CONSTRAINT == "true_positive_rate_parity":
    pred_ref_to_train, pred_ref_to = p_train, p_test
elif pred_ref_to is None:
    pred_ref_to_train, pred_ref_to = fit_threshold("true_positive_rate_parity")

## Section 6 · Compare three models (Class 2)
All three models on the same test rows. The reweighed model uses the exact weights, and the threshold model uses Fairlearn's equal-TPR constraint.

In [ ]:
# @title Compare three models, test rows
if pred_ref_rw is None:
    pred_ref_rw = fit_reweighed(exact_weights())
if pred_ref_to is None:
    pred_ref_to_train, pred_ref_to = fit_threshold("true_positive_rate_parity")

MODELS = {"original": test["predicted"].to_numpy(), "reweighed (reference)": pred_ref_rw,
          "threshold (reference)": pred_ref_to}

table, gaps, styled = compare(MODELS)
display(styled)
print(GAP_LEGEND)

In [ ]:
# @title Chart: the three gaps for the three models
fig = plot_gaps(gaps)
plt.show()

**Class 2 exit question.** Which model would you recommend for your dataset, and who pays for its errors? One sentence, with one number from Section 6.

*Datasets: Adult (Becker & Kohavi, 1996), German Credit (Hofmann, 1994) and Bank Marketing (Moro, Rita & Cortez, 2014), UCI Machine Learning Repository, CC BY 4.0; COMPAS data from ProPublica (Angwin et al., 2016). Library: Fairlearn (Weerts et al., 2023).*